In [2]:
import pandas as pd



In [3]:
df = pd.read_csv('data/complete.csv')

In [11]:
test = pd.read_csv('data/train.csv')


In [12]:
test['Is high risk'].value_counts()

Is high risk
0    28666
1      499
Name: count, dtype: int64

In [4]:
df

,Gender_F,Gender_M,Marital status_Civil marriage,Marital status_Married,Marital status_Separated,Marital status_Single / not married,Marital status_Widow,Dwelling_Co-op apartment,Dwelling_House / apartment,Dwelling_Municipal apartment,...,Has a phone_N,Has a phone_Y,Has an email_N,Has an email_Y,Income,Education level,Age,Employment length,Family member count,Is high risk
0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.000000,0.000000,1.000000,0.000000,0.417817,4.0,0.602718,0.273442,2.0,0
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.000000,0.000000,1.000000,0.000000,0.417817,1.0,0.203417,0.144410,2.0,0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.000000,0.000000,1.000000,0.000000,0.519094,4.0,0.394044,0.498453,4.0,0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.000000,0.000000,0.000000,1.000000,0.806979,1.0,0.841939,0.179319,1.0,0
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.000000,1.000000,1.000000,0.000000,0.679378,4.0,0.599130,0.044012,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46539,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.000000,0.000000,1.000000,0.000000,0.719354,4.0,0.722895,0.174940,2.0,1
46540,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.000000,0.000000,1.000000,0.000000,0.417817,4.0,0.464693,0.268228,4.0,1
46541,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.000000,0.000000,0.392192,0.607808,0.519094,1.0,0.526251,0.144322,2.0,1
46542,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.229596,0.770404,1.000000,0.000000,0.460706,4.0,0.554622,0.102809,1.0,1


In [5]:
X = df.drop(columns=['Is high risk'])
y = df['Is high risk']

In [6]:
SEED = 22

In [10]:
y.value_counts()

Is high risk
0    23272
1    23272
Name: count, dtype: int64

In [9]:
X.columns

Index(['Gender_F', 'Gender_M', 'Marital status_Civil marriage',
       'Marital status_Married', 'Marital status_Separated',
       'Marital status_Single / not married', 'Marital status_Widow',
       'Dwelling_Co-op apartment', 'Dwelling_House / apartment',
       'Dwelling_Municipal apartment', 'Dwelling_Office apartment',
       'Dwelling_Rented apartment', 'Dwelling_With parents',
       'Employment status_Commercial associate', 'Employment status_Pensioner',
       'Employment status_State servant', 'Employment status_Student',
       'Employment status_Working', 'Has a car_N', 'Has a car_Y',
       'Has a property_N', 'Has a property_Y', 'Has a work phone_N',
       'Has a work phone_Y', 'Has a phone_N', 'Has a phone_Y',
       'Has an email_N', 'Has an email_Y', 'Income', 'Education level', 'Age',
       'Employment length', 'Family member count'],
      dtype='object')

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=SEED)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((37235, 33), (9309, 33), (37235,), (9309,))

In [8]:
from imblearn.over_sampling import SMOTE
sm = SMOTE()
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

In [9]:
X_resampled.shape, len(y_resampled)

((37236, 33), 37236)

In [10]:
y_test.value_counts()

Is high risk
1    4655
0    4654
Name: count, dtype: int64

In [11]:
y_train.value_counts()

Is high risk
0    18618
1    18617
Name: count, dtype: int64

In [12]:
y_resampled.value_counts()

Is high risk
0    18618
1    18618
Name: count, dtype: int64

In [13]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

xgb_pipe = Pipeline([
    ('scaler', StandardScaler()),  # Step 1: StandardScaler for feature scaling
    ('xgb', XGBClassifier(random_state=SEED))  # Step 2: XGBoost Classifier
])

xgb_pipe.fit(X_resampled, y_resampled)
train_score = xgb_pipe.score(X_resampled, y_resampled)
print("Training Accuracy:", train_score)

Training Accuracy: 0.9890697174777098


In [14]:
xgb_pipe.score(X_test, y_test)

0.9847459447846171

In [15]:
from sklearn.metrics import f1_score, roc_curve, auc
import matplotlib.pyplot as plt


# F1 score 계산
y_pred_train = xgb_pipe.predict(X_resampled)
f1_train = f1_score(y_resampled, y_pred_train)
print("Training F1 Score:", f1_train)

y_pred_test = xgb_pipe.predict(X_test)
f1_test = f1_score(y_pred_test, y_test)
print("Test F1 Score:", f1_test)

Training F1 Score: 0.9890559036274167
Test F1 Score: 0.9847344657062997


In [16]:
from sklearn.svm import SVC

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),  # Step 1: StandardScaler for feature scaling
    ('svm', SVC(random_state=SEED))  # Step 2: Support Vector Classifier
])

svm_pipe.fit(X_resampled, y_resampled)
train_score = svm_pipe.score(X_resampled, y_resampled)
print("Training Accuracy:", train_score)

Training Accuracy: 0.9186539907616286


In [17]:
from sklearn.metrics import f1_score, roc_curve, auc
import matplotlib.pyplot as plt


# F1 score 계산
y_pred_train = svm_pipe.predict(X_resampled)
f1_train = f1_score(y_resampled, y_pred_train)
print("Training F1 Score:", f1_train)

y_pred_test = svm_pipe.predict(X_test)
f1_test = f1_score(y_pred_test, y_test)
print("Test F1 Score:", f1_test)

Training F1 Score: 0.92125308722215
Test F1 Score: 0.9139595225739492
